## Import useful libraries

In [ ]:
import os
import pickle

from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
import pyspark.sql.functions as F
from pyspark.sql.types import ArrayType, FloatType

## User settings

In [ ]:
instrument = 'EUR/USD'
granularity = 'H1'
n_back = 200
lookahead = 4

column_y = 'pd_lead'    # 'spread_close_lead', 'volatility_lead'
percentiles = [100./3., 200. / 3.]

directory_data = 'output'

In [ ]:
conf = (
    SparkConf()
    .setAppName("MyApp")
    .set("spark.executor.memory", "100G")
    .set("spark.driver.memory", "100G")
    .set("spark.driver.maxResultSize", "100G")
)

spark = SparkSession.builder.config(conf = conf).getOrCreate()

In [ ]:
training_set_filename = directory_data + '/df_training_and_testing_time_series__' + instrument.replace('/', '_') + '__' + granularity + '__' + str(n_back) + '__' + str(lookahead) + '.parquet'
training_set_filename_non_time_series = directory_data + '/df_training_and_testing_non_time_series__' + instrument.replace('/', '_') + '__' + granularity + '__' + str(n_back) + '__' + str(lookahead) + '.parquet'

In [ ]:
df_time_series = spark.read.parquet(training_set_filename)
df_time_series.show(3)

In [ ]:
columns_x = ['volatility', 'return', 'diff_spread_close', 'diff_volume', 'day_sin', 'day_cos']

In [ ]:
columns_non_time_series_to_keep = ['instrument', 'granularity', 'unix_epoch_s']
columns_non_time_series_to_keep.extend(columns_x)

In [ ]:
df_non_time_series = (
    spark
    .read
    .parquet(training_set_filename_non_time_series)
    .select(*columns_non_time_series_to_keep)
    .orderBy('instrument', 'granularity', 'unix_epoch_s')
)
df_non_time_series.show(3)

In [ ]:
df_time_series.select('instrument', 'granularity', 'unix_epoch_s', 'diff_volume').show(3)

In [ ]:
@F.udf(returnType=ArrayType(ArrayType(FloatType())))
def stack_time_series(*timeseries_list):
    result = []
    for ts in timeseries_list:
        result.append([float(x) for x in ts])
    return result

In [ ]:
column_objects = [F.col(c) for c in columns_x]

df_time_series = (
    df_time_series
    .withColumn('X', stack_time_series(*column_objects))
    .select('instrument', 'granularity', 'unix_epoch_s', column_y, 'X')
    .orderBy('instrument', 'granularity', 'unix_epoch_s')  
)

In [ ]:
df_time_series.show(3)

In [ ]:
pdf = (
    df_time_series
    .toPandas()
    .sort_values(by = ['instrument', 'granularity', 'unix_epoch_s'])
    .reset_index(drop = True)
)

In [ ]:
pdf_non_time_series = (
    df_non_time_series
    .toPandas()
    .sort_values(by = ['instrument', 'granularity', 'unix_epoch_s'])
    .reset_index(drop = True)
)

In [ ]:
#
# Load useful libraries
#
import datetime
import numpy as np

#from forex.models.model_prep_SCRATCH.BinaryClassifierResults import BinaryClassifierResults

#
# Define a base class for a classifier
#
class Base():
    
    #
    # Constructor
    #
    def __init__(
        self,
        df,
        df_non_time_series,
        instrument,
        granularity,
        timestamp_column = 'unix_epoch_s',
        randomize_rows = True,
        randomize_rows_seed = 440,
        SMOTE_seed = 38,
        class_cutoff_percentiles = [100./3., 200. / 3.],
        column_y = 'pd_lead',
        columns_x = 'X',
        columns_x_components = ['volatility', 'return', 'diff_spread_close', 'diff_volume'],
        directory_output = 'output',
    ):
        self.instrument = instrument
        self.granularity = granularity
        self.timestamp_column = timestamp_column
        self.randomize_rows = randomize_rows
        self.randomize_rows_seed = randomize_rows_seed
        self.SMOTE_seed = SMOTE_seed
        self.column_y = column_y
        self.columns_x = columns_x
        self.columns_x_components = columns_x_components
        self.directory_output = directory_output
        self.class_cutoff_percentiles = class_cutoff_percentiles

        self.df = (
            df[(df['instrument'] == self.instrument) & (df['granularity'] == self.granularity)]
            .copy()
            .sort_values(by = self.timestamp_column)
            .reset_index(drop = True)
        )

        self.df_non_time_series = (
            df_non_time_series[
                (df_non_time_series['instrument'] == self.instrument) &
                (df_non_time_series['granularity'] == self.granularity)
            ]
            .copy()
            .sort_values(by = self.timestamp_column)
            .reset_index(drop = True)
        )   

    #
    # Split data into training and test sets by timestamp ranges
    #
    def split_train_val_test_by_timestamps(
        self,
        min_timestamp_train,
        max_timestamp_train,
        min_timestamp_val,
        max_timestamp_val,
        min_timestamp_test,
        max_timestamp_test,
    ):
        df_train = (
            self.df[
                (self.df[self.timestamp_column] >= min_timestamp_train) & (self.df[self.timestamp_column] < max_timestamp_train)
            ]
            .copy()
            .sort_values(by = self.timestamp_column)
            .reset_index(drop = True)
        )

        df_val = (
            self.df[
                (self.df[self.timestamp_column] >= min_timestamp_val) & (self.df[self.timestamp_column] < max_timestamp_val)
            ]
            .copy()
            .sort_values(by = self.timestamp_column)
            .reset_index(drop = True)
        )
        
        df_test = (
            self.df[
                (self.df[self.timestamp_column] >= min_timestamp_test) & (self.df[self.timestamp_column] < max_timestamp_test)
            ]
            .copy()
            .sort_values(by = self.timestamp_column)
            .reset_index(drop = True)
        )
        return df_train, df_val, df_test

    def outcome_inator(self, df, percentiles):
        y_array = df[self.column_y].values

        outcome = []
        for y in y_array:
            if y <= percentiles[0]:
                outcome.append([1, 0, 0])
            if percentiles[0] < y and y <= percentiles[1]:
                outcome.append([0, 1, 0])
            if percentiles[1] < y:
                outcome.append([0, 0, 1])

        return outcome

    
    #
    # Discretize the dependent variable
    #
    def define_outcome(self, df_train, df_val = None, df_test = None):
        y_array = df_train[self.column_y].values
        percentiles = np.percentile(y_array, self.class_cutoff_percentiles)

        df_train['outcome'] = self.outcome_inator(df_train, percentiles)
    
        if str(type(df_val)) != "<class 'NoneType'>":
            df_val['outcome'] = self.outcome_inator(df_val, percentiles)

        if str(type(df_test)) != "<class 'NoneType'>":
            df_test['outcome'] = self.outcome_inator(df_test, percentiles)
        
        return df_train, df_val, df_test

    #
    # Calculate timestamp difference across the data set
    #
    def get_timestamp_difference(self):
        min_timestamp = np.min(self.df[self.timestamp_column])
        max_timestamp = np.max(self.df[self.timestamp_column])
        diff = max_timestamp - min_timestamp
        return min_timestamp, max_timestamp, diff

    #
    # Train on a proportion of the data (earlier in time) and test on the rest (later in time)
    #
    # This is where ROC cutoffs are computed.
    #
    def train_and_test_by_proportion_and_timestamp(self, train_val_proportion = [0.7, 0.15]):
        min_timestamp, max_timestamp, diff = self.get_timestamp_difference()
        
        timestamp_start_train = min_timestamp
        timestamp_stop_train = min_timestamp + int(train_val_proportion[0] * diff)
        
        timestamp_start_val = timestamp_stop_train
        timestamp_stop_val = timestamp_stop_train + int(train_val_proportion[1] * diff)
        
        timestamp_start_test = timestamp_stop_val
        timestamp_stop_test = max_timestamp

        df_train, df_val, df_test = self.split_train_val_test_by_timestamps(
            timestamp_start_train, timestamp_stop_train,
            timestamp_start_val, timestamp_stop_val,
            timestamp_start_test, timestamp_stop_test,
        )
            
        #print(min(df_train['unix_epoch_s']), max(df_train['unix_epoch_s']))
        #print(min(df_val['unix_epoch_s']), max(df_val['unix_epoch_s']))
        #print(min(df_test['unix_epoch_s']), max(df_test['unix_epoch_s']))
        #print()
        
        df_train, df_val, df_test = self.define_outcome(df_train, df_val = df_val, df_test = df_test)

        X_train = np.array([np.array(i) for i in df_train['X'].to_numpy()])
        X_val = np.array([np.array(i) for i in df_val['X'].to_numpy()])
        X_test = np.array([np.array(i) for i in df_test['X'].to_numpy()])

        #print(X_train.shape, X_val.shape, X_test.shape)

        # normalize
        X_train_non_time_series = (
            self.df_non_time_series[self.df_non_time_series[self.timestamp_column] < timestamp_stop_train]
            [self.columns_x_components]
            .to_numpy()
        )
        the_mean_m_set = np.mean(X_train_non_time_series, axis = 0)
        the_std_m_set = np.std(X_train_non_time_series, axis = 0)
        m, n = X_train[0, :, :].shape
        the_mean = np.zeros([m, n])
        the_std = np.zeros([m, n])
        for i in range(0, m):
            the_mean[i, :] = the_mean_m_set[i]
            the_std[i, :] = the_std_m_set[i]
        X_train_norm = (X_train - the_mean) / the_std
        X_val_norm = (X_val - the_mean) / the_std
        X_test_norm = (X_test - the_mean) / the_std

        # print(X_train_norm.shape)

        y_train = np.array([np.array(y) for y in df_train['outcome'].to_numpy()])
        y_val = np.array([np.array(y) for y in df_val['outcome'].to_numpy()])
        y_test = np.array([np.array(y) for y in df_test['outcome'].to_numpy()])
        
        #print(y_train.shape)
        #print(y_val.shape)
        #print(y_test.shape)
        #print()

        #to_pickle = {
        #    'X_train_norm' : X_train_norm,
        #    'X_val_norm' : X_val_norm,
        #    'X_test_norm' : X_test_norm,
        #    'y_train' : y_train,
        #    'y_val' : y_val,
        #    'y_test' : y_test,
        #    'normalization_mean' : the_mean_m_set,
        #    'normalization_std' : the_std_m_set,
        #}

        to_pickle = {
            'train' : {
                'M' : X_train_norm,
                'y' : y_train,
            },
            'val' : {
                'M' : X_val_norm,
                'y' : y_val,
            },
            'test' : {
                'M' : X_test_norm,
                'y' : y_test,
            },
        }
        with open(self.directory_output + '/data.pickled', 'wb') as f:
            pickle.dump(to_pickle, f)





        #tpr, fpr, thresholds = self.train_it_then_test_it(df_train, df_test)
        #self.train_and_test_by_proportion_and_timestamp_results = BinaryClassifierResults(tpr, fpr, thresholds)
        #self.train_and_test_by_proportion_and_timestamp_results.fit()7_

In [ ]:
b = Base(pdf, pdf_non_time_series, instrument, granularity, columns_x_components = columns_x)
b.train_and_test_by_proportion_and_timestamp()

In [ ]:
with open('output/data.pickled', 'rb') as f:
    data = pickle.load(f)

In [ ]:
data